# 06 · Preview das análises do dashboard

Uma amostra, com Plotly, do que o dashboard Streamlit mostra de forma interativa. Para ver o resultado
sem subir o serviço no Docker

In [1]:
# Configuracao inicial
import os
import warnings
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore")


def find_project_root():
    current = Path.cwd().resolve()
    for p in [current] + list(current.parents):
        if (p / "src").exists() and (p / "data").exists() and (p / "config").exists():
            return p
    raise RuntimeError("Raiz do projeto nao encontrada")


def project_path(*segments):
    return find_project_root().joinpath(*segments)


root = find_project_root()
os.chdir(root)
print(f"Diretorio de trabalho: {root}")


Diretorio de trabalho: C:\Users\user\Downloads\Códigos\olist-ecommerce-pipeline\Template


## Receita mensal

In [2]:
import plotly.express as px
from src.etl.db import get_engine

engine = get_engine()
df = pd.read_sql(
    """
    SELECT d.year, d.month, SUM(f.price + f.freight_value) AS receita
    FROM dw.fact_order_items f
    JOIN dw.dim_date d ON f.order_purchase_date_key = d.date_key
    GROUP BY d.year, d.month ORDER BY d.year, d.month
    """,
    engine,
)
df["periodo"] = pd.to_datetime(dict(year=df["year"], month=df["month"], day=1))
px.line(df, x="periodo", y="receita", markers=True)

## Top 10 categorias por receita

In [3]:
top_cat = pd.read_sql(
    """
    SELECT p.product_category_name_english AS categoria, SUM(f.price) AS receita
    FROM dw.fact_order_items f
    JOIN dw.dim_products p ON f.product_key = p.product_key
    GROUP BY categoria ORDER BY receita DESC LIMIT 10
    """,
    engine,
)
px.bar(top_cat.sort_values("receita"), x="receita", y="categoria", orientation="h")

## Distribuição de notas de avaliação

In [4]:
reviews = pd.read_sql("SELECT review_score FROM dw.fact_reviews WHERE review_score IS NOT NULL", engine)
counts = reviews["review_score"].value_counts().sort_index().reset_index()
counts.columns = ["nota", "quantidade"]
px.bar(counts, x="nota", y="quantidade")